# DevOps Interview Lab (Google Colab)

Easy to medium commands, one short command at a time, in the same order as the lab page. Each topic has **Level 1 Basic**, **Level 2 Next level** and **Level 3 Medium**, then interview questions.

**How to use**
1. Run the first code cell below (it creates the `%%b` shell and the sample files).
2. Go to a topic in the table of contents (left sidebar) and run the cells **from top to bottom**. Each step has a short explanation above its command.
3. A level can be repeated: run its reset cell at the end of the topic.

**Keep a copy:** *File > Save a copy in Drive*. The latest version is always at https://github.com/shr1-live/devops (folder `colab`).

**Good to know**
- Colab gives you a real Linux machine with root, so `apt`, `nginx`, `git` and `terraform` all work here.
- **Docker does not run inside Colab.** Topic 5 is a reference, and you practise it in the lab page on Docker Desktop. Topic 8 (Cloud) needs a cloud account, so it is a reference too.
- The machine is wiped when the Colab session ends. That is fine: re-run the first cell and carry on.

In [ ]:
# Run this cell first (Runtime > Run all is also fine for this one).
# It gives you the %%b cell magic: every %%b cell runs in ONE persistent Linux shell, just like a terminal.
# So cd, variables, export and background jobs (&) stay alive from one cell to the next.
import os, json, uuid, pathlib, subprocess
from IPython.core.magic import register_cell_magic

FILES = json.loads(r'''{"/scripts/lab-files/access.log":"10.0.0.1 GET /home 200\n10.0.0.2 GET /home 200\n10.0.0.1 GET /api 500\n10.0.0.3 GET /api 200\n10.0.0.1 GET /login 404\n10.0.0.2 POST /login 200\n10.0.0.4 GET /home 200\n10.0.0.1 GET /api 500\n","/scripts/lab-files/bad.conf":"server {\n  listen 8082;\n  location / { proxy_pass http://127.0.0.1:9999; }\n}\n","/scripts/lab-files/check.sh":"#!/bin/bash\nif [ -f \"$1\" ]; then\n  echo \"found $1\"\nelse\n  echo \"missing $1\"\n  exit 1\nfi\n","/scripts/lab-files/func.sh":"#!/bin/bash\ngreet() {\n  echo \"Hi $1\"\n}\ngreet Asha\ngreet Ravi\n","/scripts/lab-files/hello.sh":"#!/bin/bash\nname=${1:-world}\necho \"Hello, $name\"\n","/scripts/lab-files/lb.conf":"upstream pool {\n  server 127.0.0.1:9001;\n  server 127.0.0.1:9002;\n}\nserver {\n  listen 9001;\n  location / { return 200 \"backend 1\\n\"; }\n}\nserver {\n  listen 9002;\n  location / { return 200 \"backend 2\\n\"; }\n}\nserver {\n  listen 8081;\n  location / { proxy_pass http://pool; }\n}\n","/scripts/lab-files/loop.sh":"#!/bin/bash\nfor i in 1 2 3; do\n  echo \"round $i\"\ndone\n","/scripts/lab-files/loose.sh":"#!/bin/bash\necho \"start\"\ncat missing-file.txt\necho \"never printed\"\n","/scripts/lab-files/proxy.conf":"server {\n  listen 9000;\n  location / { return 200 \"backend ok\\n\"; }\n}\nserver {\n  listen 8080;\n  location / { proxy_pass http://127.0.0.1:9000; }\n}\n","/scripts/lab-files/routes.conf":"server {\n  listen 8083;\n  location /api/ { return 200 \"api\\n\"; }\n  location = /health { return 200 \"healthy\\n\"; }\n  location / { return 200 \"home\\n\"; }\n}\n","/scripts/lab-files/site.conf":"server {\n  listen 8000;\n  root /var/www/lab;\n  index index.html;\n}\n","/scripts/lab-files/strict.sh":"#!/bin/bash\nset -euo pipefail\necho \"start\"\ncat missing-file.txt\necho \"never printed\"\n","/work/lab-ci/ci.yml":"name: ci\non:\n  push:\n    branches: [main]\n  pull_request:\njobs:\n  build:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Checkout code\n        uses: actions/checkout@v4\n      - name: Run tests\n        run: echo \"tests passed\"\n      - name: Build image\n        run: docker build -t app:${{ github.sha }} .\n","/work/lab-tf/basic.tf":"variable \"name\" {\n  default = \"DevOps\"\n}\n\nresource \"terraform_data\" \"hello\" {\n  input = \"Hello, ${var.name}\"\n}\n\noutput \"greeting\" {\n  value = terraform_data.hello.input\n}\n","/work/lab-tf/mod/main.tf":"module \"greeter\" {\n  source = \"./modules/greeter\"\n  names  = [\"asha\", \"ravi\", \"meera\"]\n}\n\noutput \"all\" {\n  value = module.greeter.messages\n}\n","/work/lab-tf/mod/modules/greeter/main.tf":"variable \"names\" {\n  type = list(string)\n}\n\nresource \"terraform_data\" \"msg\" {\n  for_each = toset(var.names)\n  input    = \"hi ${each.key}\"\n}\n\noutput \"messages\" {\n  value = [for m in terraform_data.msg : m.input]\n}\n","/work/lab-docker/Dockerfile":"FROM alpine:3.20\nRUN apk add --no-cache curl\nCMD [\"echo\", \"hello from my image\"]\n"}''')
for dest, text in FILES.items():
    p = pathlib.Path(dest); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text)
    if dest.endswith(".sh"): p.chmod(0o755)

_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive", PS1="")
_shell = None
def _start():
    global _shell
    _shell = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=os.path.expanduser("~"), env=_env)

@register_cell_magic
def b(line, cell):
    """Run the cell in the persistent bash session. '%%b reset' starts a fresh shell."""
    global _shell
    if line.strip() == "reset" or _shell is None or _shell.poll() is not None:
        if _shell is not None and _shell.poll() is None: _shell.kill()
        _start()
    tag = "__END_" + uuid.uuid4().hex + "__"
    _shell.stdin.write(cell.rstrip("\n") + "\n" + "echo " + tag + "\n"); _shell.stdin.flush()
    for out in _shell.stdout:
        if tag in out:
            before = out.split(tag)[0]
            if before: print(before, end="", flush=True)
            break
        print(out, end="", flush=True)

_start()
print("Ready. Sample files are in /scripts/lab-files, /work/lab-ci, /work/lab-tf, /work/lab-docker.")

# 1 · Linux

The commands every Linux interview starts with: files and folders first, then searching, permissions, disk and processes, then users, links and archives.

## Level 1 · Basic

**1. Make a practice folder**

mkdir makes a folder. -p means no error if it already exists. ~ is your home folder. All practice stays inside ~/lab, so nothing else is touched.

In [ ]:
%%b
mkdir -p ~/lab

**2. Go into it**

cd = change directory. cd alone goes home, cd .. goes up one level, cd - goes back to the previous folder.

In [ ]:
%%b
cd ~/lab

**3. Where am I?**

pwd = print working directory: the full path of the folder you are in. Interview: 'how do you know where you are?'

In [ ]:
%%b
pwd

**4. Who am I?**

Prints your user name. root is the all-powerful admin. On real servers you are a normal user and put sudo in front of admin commands.

In [ ]:
%%b
whoami

**5. Create an empty file**

touch makes an empty file. If the file already exists it only updates its time.

In [ ]:
%%b
touch a.txt

**6. List the files**

ls lists the folder. Common versions: ls -l (details), ls -a (also hidden files that start with a dot), ls -lh (sizes in KB/MB).

In [ ]:
%%b
ls

**7. Write a line into the file**

echo prints text. > sends it into a file and REPLACES what was there. >> adds to the end instead. Interview: 'difference between > and >>'.

In [ ]:
%%b
echo hello > a.txt

**8. Read the file**

cat prints the whole file. For long files use less (scroll, q to quit), head (first lines) or tail (last lines).

In [ ]:
%%b
cat a.txt

**9. Copy it**

cp source destination. Add -r to copy a whole folder.

In [ ]:
%%b
cp a.txt b.txt

**10. Make a folder**

mkdir NAME makes one folder. With -p it also makes parent folders: mkdir -p a/b/c.

In [ ]:
%%b
mkdir notes

**11. Move the copy into it**

mv moves AND renames. mv old.txt new.txt renames. There is no separate rename command.

In [ ]:
%%b
mv b.txt notes

**12. List with details**

-l = long format: permissions, owner, group, size, date, name. First letter d = folder, - = file, l = link.

In [ ]:
%%b
ls -l

**13. Delete a file**

rm deletes for good, there is no recycle bin. Interview: 'what does rm -rf / do?' It wipes the system. Never run it.

In [ ]:
%%b
rm a.txt

**14. Delete a folder**

-r = recursive, needed for folders and everything inside. rmdir only removes EMPTY folders.

In [ ]:
%%b
rm -r notes

**15. Date and time**

Prints the current date and time of the machine. Logs use this clock, so a wrong clock confuses debugging.

In [ ]:
%%b
date

## Level 2 · Next level

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Copy a sample web log here**

The dot means 'this folder'. The log has 8 lines: client IP, method, page, status code.

In [ ]:
%%b
cp /scripts/lab-files/access.log .

**2. First lines of a file**

head shows the first lines (10 by default). The number after the dash sets how many.

In [ ]:
%%b
head -3 access.log

**3. Last lines of a file**

tail shows the last lines. tail -f FILE follows a log live as it grows, Ctrl+C stops. Interview: 'how do you watch a log in real time?'

In [ ]:
%%b
tail -2 access.log

**4. Count lines**

wc = word count. -l counts lines, -w words, -c bytes. Quick way to count requests or users.

In [ ]:
%%b
wc -l access.log

**5. Search inside a file**

grep prints the lines containing the text. Useful flags: -i ignore case, -v lines that do NOT match, -n show line numbers, -r search a whole folder.

In [ ]:
%%b
grep 500 access.log

**6. Count the matches**

-c prints how many lines matched: here 5 successful requests.

In [ ]:
%%b
grep -c 200 access.log

**7. Find a file by name**

find WHERE -name PATTERN. Quotes stop the shell from expanding the *. Interview: find a file anywhere: find / -name nginx.conf

In [ ]:
%%b
find . -name '*.log'

**8. Make the file private**

chmod changes permissions. Each digit is one group: owner, group, others. r=4, w=2, x=1, add them up. 6 = read+write, 0 = nothing.

In [ ]:
%%b
chmod 600 access.log

**9. Check the permissions**

-rw------- = only the owner can read and write. Common values: 755 (programs, folders), 644 (normal files), 600 (private files), 700 (private folders).

In [ ]:
%%b
ls -l access.log

**10. Disk space of the machine**

df = disk free for each disk. -h = human sizes. Check Use% and 'Mounted on'. Interview: 'the disk is full, what is your first command?'

In [ ]:
%%b
df -h

**11. Disk space of this folder**

du = disk usage of files. -s = one total, -h = human sizes. df tells you WHICH disk is full, du tells you WHICH folder is eating it.

In [ ]:
%%b
du -sh .

**12. Memory**

free shows RAM in MB. Read the 'available' column, not 'free': Linux uses spare RAM as cache, so 'free' always looks small on a healthy machine.

In [ ]:
%%b
free -m

**13. List all processes**

ps aux = every running process with user, PID (process id), CPU%, memory% and command. You need the PID to stop a process.

In [ ]:
%%b
ps aux

**14. Start a background process**

& runs the command in the background and gives your prompt back. It prints [job number] and the PID. sleep 300 simply waits 5 minutes.

In [ ]:
%%b
sleep 300 &

**15. Find its PID**

pgrep prints the PIDs of processes with that name. Same idea as ps aux | grep sleep, but shorter.

In [ ]:
%%b
pgrep sleep

**16. Stop it**

pkill sends a polite stop signal (SIGTERM) by name. kill PID does it by number. kill -9 PID (SIGKILL) forces it when the polite one is ignored.

In [ ]:
%%b
pkill sleep

## Level 3 · Medium

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Copy the log again**

Every level starts from fresh files, so you can do the levels in any order.

In [ ]:
%%b
cp /scripts/lab-files/access.log .

**2. Create a user**

useradd makes an account. Real use: useradd -m -s /bin/bash NAME (-m makes a home folder, -s sets the shell). Accounts are stored in /etc/passwd.

In [ ]:
%%b
useradd dev1

**3. Check the user**

id shows the user id (uid), main group (gid) and every group the user is in.

In [ ]:
%%b
id dev1

**4. See the account line**

Colon separated: name, x (the password hash is in /etc/shadow), uid, gid, comment, home folder, shell.

In [ ]:
%%b
grep dev1 /etc/passwd

**5. Give admin rights**

usermod changes a user. -aG = ADD to a group. Forgetting the -a REPLACES all their groups, a classic mistake. Members of the sudo group may run admin commands.

In [ ]:
%%b
usermod -aG sudo dev1

**6. Check the groups**

groups lists the groups of a user. dev1 now shows sudo.

In [ ]:
%%b
groups dev1

**7. Make a shortcut (soft link)**

ln -s makes a symbolic link: a small file that points to another path. Without -s you get a hard link: a second name for the same data.

In [ ]:
%%b
ln -s /etc/hostname mylink

**8. Check the link**

The arrow shows the target and the first letter l means link. Delete the target and a soft link breaks, a hard link keeps working.

In [ ]:
%%b
ls -l mylink

**9. Pack a file into an archive**

tar c = create, z = gzip compress, f = archive file name. Keep the letters together, then the archive name, then the files.

In [ ]:
%%b
tar czf logs.tgz access.log

**10. Look inside the archive**

t = list the contents without unpacking. To unpack: tar xzf logs.tgz (x = extract).

In [ ]:
%%b
tar tzf logs.tgz

**11. Where does the shell look for programs?**

$PATH is a list of folders separated by colons. When you type ls the shell searches them in order. 'command not found' means the program is in none of them.

In [ ]:
%%b
echo $PATH

**12. Set an environment variable**

export makes a variable visible to programs you start. Without export it stays inside your shell. To keep it after logout, add the line to ~/.bashrc.

In [ ]:
%%b
export APP=demo

**13. Check it**

env lists all environment variables. The pipe | sends env's output into grep, which keeps only the lines that match.

In [ ]:
%%b
env | grep APP

**14. Is a package installed?**

dpkg -l NAME shows the package (ii = installed). Install with apt-get install NAME, remove with apt-get remove NAME.

In [ ]:
%%b
dpkg -l curl

**15. Where is a command stored?**

which shows the full path of the program that would run. Useful when two versions are installed.

In [ ]:
%%b
which ls

**16. Is the machine busy?**

Shows how long the machine is up and the load average for 1, 5 and 15 minutes. If the load is higher than the number of CPU cores, the machine is overloaded.

In [ ]:
%%b
uptime

**17. Remove the user**

userdel deletes the account (add -r to delete the home folder too).

In [ ]:
%%b
userdel dev1

## On a real server (reference)

Containers have no systemd, so these are copy-only. Run them on a real Linux server or a KodeKloud playground.

**1. Is a service running?**

Shows active (running) or failed, plus the last log lines. First thing to check when a site is down.

```bash
systemctl status nginx
```

**2. Start, stop, restart**

start / stop / restart / reload. reload re-reads the config without dropping connections.

```bash
systemctl restart nginx
```

**3. Start at boot**

enable = start automatically after a reboot. disable turns it off. enable --now does both enable and start.

```bash
systemctl enable nginx
```

**4. Why did it fail? Read its logs**

journalctl reads the system journal. -u = one service, -n 30 = last 30 lines, -f = follow live.

```bash
journalctl -u nginx -n 30
```

**5. List failed services**

Shows every service that crashed. Quick health check after a reboot.

```bash
systemctl --failed
```

**6. Edit the schedule (cron)**

Opens your cron table. A line looks like: */5 * * * * /home/me/backup.sh (minute hour day month weekday command).

```bash
crontab -e
```

**7. Edit sudo rules safely**

Always edit /etc/sudoers with visudo. It checks the syntax before saving, a typo cannot lock you out.

```bash
visudo
```

## Interview Q&A

**1. The disk is full. How do you find out why?**

- Step 1: df -h shows which disk is full.
- Step 2: du -sh /var/* shows which folder is the biggest. Repeat one level deeper until you find it.
- Also check: df -i. A disk with free space can still be 'full' if all inodes (file slots) are used by millions of tiny files.

**2. A process uses too much CPU. What do you do?**

- Find it: top (press P to sort by CPU) or ps aux --sort=-%cpu | head.
- Stop it: kill PID first (polite). If it ignores that: kill -9 PID.
- Then: Find out WHY: check its logs, do not just keep killing it.

**3. How do you find a file, and text inside files?**

- File by name: find / -name nginx.conf
- Text inside files: grep -r 'error' /var/log
- Tip: Add 2>/dev/null to hide 'Permission denied' noise.

**4. What do 755, 644 and 600 mean?**

- 755: rwxr-xr-x: owner full, everyone else read and run. For programs and folders.
- 644: rw-r--r--: owner read and write, everyone else read. Normal files.
- 600: rw-------: only the owner. Private keys (ssh refuses keys that are more open).

**5. Hard link vs soft (symbolic) link?**

- Soft: ln -s target link. A pointer to a PATH. Breaks if the target is deleted. Can cross disks.
- Hard: ln target link. Another NAME for the same data. Survives deleting the original name.

**6. What is the difference between >, >> and |?**

- >: Sends output to a file and REPLACES it.
- >>: Sends output to a file and ADDS to the end.
- |: Sends the output of one command into the next command.

**7. How do you check CPU, memory and load?**

- CPU and processes: top
- Memory: free -m (read 'available')
- Load: uptime (compare with the number of cores: nproc)

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
rm -rf ~/lab
userdel dev1

# 2 · Bash scripting

From running a script and reading its exit code, to if and loops, to the log-analysis one-liners interviewers love. Pipes are built up one piece at a time.

## Level 1 · Basic

**1. Make a practice folder**

All script practice happens in ~/lab.

In [ ]:
%%b
mkdir -p ~/lab

**2. Go into it**

cd = change directory.

In [ ]:
%%b
cd ~/lab

**3. Copy a ready-made script**

The script has 3 lines. You will read it next.

In [ ]:
%%b
cp /scripts/lab-files/hello.sh .

**4. Read the script**

Line 1 #!/bin/bash (the shebang) says which program runs the file. Line 2 sets name to the first argument ($1), or 'world' if none. Line 3 prints it.

In [ ]:
%%b
cat hello.sh

**5. Look at its permissions**

The x in -rwxr-xr-x means 'may be executed'. A script without x cannot be started with ./script.sh.

In [ ]:
%%b
ls -l hello.sh

**6. Make it executable**

+x adds the execute permission for everyone. Interview: 'I wrote a script and get Permission denied, why?' You forgot chmod +x.

In [ ]:
%%b
chmod +x hello.sh

**7. Run it**

./ means 'in this folder'. Without it the shell only searches $PATH and says command not found.

In [ ]:
%%b
./hello.sh

**8. Run it with an argument**

Words after the script name are arguments. Inside the script the first is $1, the second $2, how many is $#, all of them is $@.

In [ ]:
%%b
./hello.sh Asha

**9. Run it with bash directly**

bash FILE runs the script without needing +x. Handy for a quick test.

In [ ]:
%%b
bash hello.sh Ravi

**10. Exit code of the last command**

$? holds the exit code of the previous command. 0 = success, anything else = failure. Scripts and CI pipelines decide pass or fail from this number.

In [ ]:
%%b
echo $?

**11. Set a variable**

NO spaces around =. name = Meera fails, because the shell thinks name is a command.

In [ ]:
%%b
name=Meera

**12. Use the variable**

$name gives the value. Use quotes when it may contain spaces: echo "$name".

In [ ]:
%%b
echo $name

**13. Quotes: double expands, single does not**

Double quotes expand variables. Single quotes print the text exactly as typed: echo 'Hello $name' prints Hello $name.

In [ ]:
%%b
echo "Hello $name"

## Level 2 · Next level

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Copy a script with an if**

check.sh tests whether the file given as the first argument exists.

In [ ]:
%%b
cp /scripts/lab-files/check.sh .

**2. Copy a script with a loop**

loop.sh repeats a command three times.

In [ ]:
%%b
cp /scripts/lab-files/loop.sh .

**3. Read the if script**

if [ -f FILE ] is true when the file exists. Other tests: -d folder, -z empty text, -eq number equal. Spaces inside [ ] are required.

In [ ]:
%%b
cat check.sh

**4. Test: the file exists**

The if branch runs and prints found.

In [ ]:
%%b
bash check.sh hello.sh

**5. Test: the file is missing**

The else branch runs, and the script ends with exit 1 to tell the caller it failed.

In [ ]:
%%b
bash check.sh nothing.txt

**6. Read the exit code**

1 = the script failed on purpose. Without a final exit the script returns the exit code of its last command.

In [ ]:
%%b
echo $?

**7. Run the loop**

for i in 1 2 3 repeats the body once per value. Also common: while read line, and for f in *.log.

In [ ]:
%%b
bash loop.sh

**8. Copy a sample log**

Columns: IP, method, page, status.

In [ ]:
%%b
cp /scripts/lab-files/access.log .

**9. Pick one column**

cut -d' ' sets the separator to a space, -f1 picks field 1 (the IP). Try -f4 for the status code.

In [ ]:
%%b
cut -d' ' -f1 access.log

**10. Count the GET requests**

grep -c counts matching lines.

In [ ]:
%%b
grep -c GET access.log

**11. Lines that are NOT 200**

-v inverts the match: shows everything except 200. First thing to do when looking for failed requests.

In [ ]:
%%b
grep -v 200 access.log

**12. Count all lines**

Total requests = number of lines.

In [ ]:
%%b
wc -l access.log

## Level 3 · Medium

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Copy the log again**

Fresh start for this level.

In [ ]:
%%b
cp /scripts/lab-files/access.log .

**2. Save the IP column to a file**

> saves output into a file. Now ips.txt has one IP per line. Next we turn it into a 'top clients' list, one piece at a time.

In [ ]:
%%b
cut -d' ' -f1 access.log > ips.txt

**3. Piece 1: sort**

sort puts identical lines next to each other. uniq only merges NEIGHBOURING lines, so you must sort first.

In [ ]:
%%b
sort ips.txt

**4. Piece 2: count the repeats**

uniq -c puts a count in front of each distinct line. 10.0.0.1 appears 4 times.

In [ ]:
%%b
sort ips.txt | uniq -c

**5. Piece 3: biggest first**

sort -rn = numeric, reverse. Add | head -3 for the top 3. This exact pipeline is a very common interview question.

In [ ]:
%%b
sort ips.txt | uniq -c | sort -rn

**6. awk: print the status column**

awk splits each line into fields: $1 first column, $4 fourth, $NF last. print shows them.

In [ ]:
%%b
awk '{print $4}' access.log

**7. awk: filter rows**

Only the lines where column 4 equals 500. Server errors. Use $4 >= 500 for all 5xx.

In [ ]:
%%b
awk '$4 == 500' access.log

**8. sed: replace text**

sed 's/old/new/' replaces the first match on every line and prints the result. The file is NOT changed. Add -i to edit in place.

In [ ]:
%%b
sed 's/GET/get/' access.log

**9. Copy the strict and loose scripts**

loose.sh prints start, tries to read a missing file, then prints never printed.

In [ ]:
%%b
cp /scripts/lab-files/loose.sh .

**10. Copy the strict script**

strict.sh is the same script with one extra line: set -euo pipefail.

In [ ]:
%%b
cp /scripts/lab-files/strict.sh .

**11. Run the loose script**

By default bash keeps going after an error, so 'never printed' IS printed. Dangerous in deploy scripts.

In [ ]:
%%b
bash loose.sh

**12. Run the strict script**

set -e stops at the first error, -u stops on an unset variable, -o pipefail fails a pipe if any part fails. Interview: 'what does set -euo pipefail do?'

In [ ]:
%%b
bash strict.sh

**13. Copy a script with a function**

A function groups commands so you can reuse them.

In [ ]:
%%b
cp /scripts/lab-files/func.sh .

**14. Run it**

greet Asha calls the function and Asha becomes $1 inside it.

In [ ]:
%%b
bash func.sh

**15. Check syntax without running**

-n reads the script and reports syntax errors but runs nothing. No output means no syntax errors.

In [ ]:
%%b
bash -n hello.sh

## Cron and background jobs (reference)

Needs cron on a real server. Copy-only.

**1. List your scheduled jobs**

Shows the cron table of the current user.

```bash
crontab -l
```

**2. Edit the schedule**

Line format: minute hour day-of-month month day-of-week command. */5 * * * * = every 5 minutes. 0 2 * * * = every day at 02:00.

```bash
crontab -e
```

**3. Log the output of a cron job**

>> appends the output. 2>&1 sends errors into the same file. Without this, cron output is lost.

```bash
0 2 * * * /home/me/backup.sh >> /var/log/backup.log 2>&1
```

**4. Keep a script running after logout**

nohup ignores the hangup signal when you close the terminal. & puts it in the background.

```bash
nohup ./job.sh &
```

**5. Debug a script**

-x prints every command before it runs. Best way to see where a script goes wrong.

```bash
bash -x script.sh
```

## Interview Q&A

**1. What does set -euo pipefail do?**

- -e: Stop at the first command that fails.
- -u: Treat an unset variable as an error.
- -o pipefail: A pipeline fails if ANY command in it fails, not only the last.

**2. What is an exit code?**

- Meaning: A number every command returns. 0 = success, non-zero = failure.
- Read it: echo $? right after the command.
- Use it: if command; then ... fi, or command || exit 1.

**3. Single quotes vs double quotes?**

- Double: Variables and $(commands) are expanded.
- Single: Everything is literal text.

**4. How do you pass and read arguments?**

- Read: $1 $2 ... are the arguments, $# how many, $@ all of them, $0 the script name.

**5. Find the IPs that hit the server most**

- Answer: cut -d' ' -f1 access.log | sort | uniq -c | sort -rn | head
- Why sort first?: uniq only merges neighbouring lines.

**6. Read the cron schedule */15 9-17 * * 1-5**

- Meaning: Every 15 minutes, between 09:00 and 17:59, Monday to Friday.
- Order: minute, hour, day of month, month, day of week.

**7. Difference between ; && and ||?**

- ;: Run the next command always.
- &&: Run the next command only if the first SUCCEEDED.
- ||: Run the next command only if the first FAILED.

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
rm -rf ~/lab

# 3 · Networking

From 'what is my IP' and ping, to DNS and open ports, to who is listening on a port. Everything in Levels 1-3 works offline. The last tab needs internet.

## Level 1 · Basic

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Setup: refresh the package list**

Downloads the list of available packages. -qq = quiet. Needs internet the first time. Skip it if the tools below are already installed.

In [ ]:
%%b
apt-get update -qq

**2. Setup: install the network tools**

ping, dig/nslookup, nc, traceroute and netstat are not in a minimal Ubuntu. -y answers yes for you.

In [ ]:
%%b
apt-get install -y -qq iputils-ping dnsutils netcat-openbsd traceroute net-tools

**3. Make a practice folder**

Keeps the practice in one place.

In [ ]:
%%b
mkdir -p ~/lab

**4. Name of this machine**

The machine name. In scripts and logs it tells you which server you are on.

In [ ]:
%%b
hostname

**5. Show IP addresses**

ip addr (short: ip a) lists every network interface with its IP. lo = loopback (127.0.0.1, this machine itself), eth0 = the real network card. Older name: ifconfig.

In [ ]:
%%b
ip addr

**6. Show the routes**

The routing table. 'default via X' = where packets go when no other rule matches: your gateway (router).

In [ ]:
%%b
ip route

**7. Local name list**

/etc/hosts maps names to IPs on this machine only. It is checked before DNS, so it is a quick way to override a name.

In [ ]:
%%b
cat /etc/hosts

**8. Which DNS server is used?**

Lists the DNS servers this machine asks to turn names into IPs.

In [ ]:
%%b
cat /etc/resolv.conf

**9. Ping this machine**

ping sends small test packets and times the answer. -c 2 = only two (otherwise it never stops, Ctrl+C ends it). 0% packet loss = reachable.

In [ ]:
%%b
ping -c 2 127.0.0.1

**10. Is curl there?**

curl is the tool for talking HTTP from the terminal. You will use it a lot: -I headers only, -s silent, -v details, -L follow redirects, -o save to a file.

In [ ]:
%%b
curl --version

## Level 2 · Next level

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. DNS lookup (easy output)**

nslookup asks DNS for the IP of a name. The first Address is the DNS server, the next ones are the answer.

In [ ]:
%%b
nslookup example.com

**2. DNS lookup (short answer)**

dig is the admin's DNS tool. +short prints only the answer. Try dig example.com MX for mail servers and NS for name servers.

In [ ]:
%%b
dig example.com +short

**3. Name to IP on this machine**

getent uses the same lookup order as programs do: /etc/hosts first, then DNS.

In [ ]:
%%b
getent hosts localhost

**4. Which ports are listening?**

ss lists sockets. -t TCP, -l listening only, -n numbers (do not resolve names). Add -p to see the process. This replaces the older netstat.

In [ ]:
%%b
ss -tln

**5. Start a listener on port 9000**

nc (netcat) is the network Swiss army knife. -l waits for a connection. & keeps it in the background. We now have a tiny 'server' to test against.

In [ ]:
%%b
nc -l 9000 &

**6. Is it listening?**

Port 9000 is in the list. This is how you prove a service is up.

In [ ]:
%%b
ss -tln

**7. Test the port**

-z = just connect, -v = tell me the result. 'succeeded' = open. 'refused' = nothing listens. A hang = a firewall drops the packets.

In [ ]:
%%b
nc -zv 127.0.0.1 9000

**8. Stop the listener**

Clean up the background process.

In [ ]:
%%b
pkill nc

## Level 3 · Medium

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Compact address list**

-br = brief, one line per interface with its state. Easy to read when there are many interfaces.

In [ ]:
%%b
ip -br addr

**2. Interface details**

Shows state (UP/DOWN), MTU and MAC address. A DOWN interface is the first thing to check when 'the network is dead'.

In [ ]:
%%b
ip link

**3. Socket summary**

How many TCP connections exist and in which state. Many TIME-WAIT or SYN-RECV entries can point to a problem.

In [ ]:
%%b
ss -s

**4. Start a listener again**

A small server on port 9000.

In [ ]:
%%b
nc -l 9000 &

**5. Who owns the port?**

-p adds the process that owns each port. Interview: 'what is using port 80?' Answer: ss -tlnp | grep :80 (or lsof -i :80).

In [ ]:
%%b
ss -tlnp

**6. Connect to it**

A successful port test. The same command against a real server tells you whether a firewall blocks the port.

In [ ]:
%%b
nc -zv 127.0.0.1 9000

**7. Stop the listener**

Clean up.

In [ ]:
%%b
pkill nc

**8. Trace the path**

traceroute shows every router (hop) on the way to a target. -m 3 stops after 3 hops. Stars (* * *) mean a hop does not answer, not necessarily a failure. Here the target is this machine: one hop.

In [ ]:
%%b
traceroute -m 3 127.0.0.1

## Needs internet (optional) (optional)

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Older command for open ports (needs net-tools)**

netstat shows the same list as ss. Interviewers still ask for it, so know both. It comes from the net-tools package installed in the setup step.

In [ ]:
%%b
netstat -tln

**2. Ping the internet**

8.8.8.8 is Google's DNS. Works = you have internet by IP. Interview: 'ping works by IP but not by name' means DNS is broken.

In [ ]:
%%b
ping -c 2 8.8.8.8

**3. Ping by name**

Needs DNS first, then the ping.

In [ ]:
%%b
ping -c 2 example.com

**4. Fetch only the header**

-I shows the response headers. 200 = OK, 301/302 = redirect, 404 = not found, 500 = server error.

In [ ]:
%%b
curl -I https://example.com

**5. Download the page**

-s = silent, no progress bar. Add -L to follow redirects.

In [ ]:
%%b
curl -s https://example.com

**6. Is a remote port open?**

Port 443 = HTTPS. 'succeeded' = reachable.

In [ ]:
%%b
nc -zv example.com 443

**7. See the TLS handshake**

-v shows DNS, TCP connect, TLS version and certificate. This is how you debug HTTPS problems. -o /dev/null throws the page away.

In [ ]:
%%b
curl -sv https://example.com -o /dev/null

**8. Reverse DNS**

-x turns an IP back into a name.

In [ ]:
%%b
dig -x 8.8.8.8 +short

**9. Trace the path to the internet**

Each line is a router on the way.

In [ ]:
%%b
traceroute -m 5 8.8.8.8

## Firewall and packet capture (reference)

Needs a real server with root access. Copy-only.

**1. Firewall status (Ubuntu)**

ufw is the simple firewall front end. Shows allowed ports.

```bash
ufw status
```

**2. Allow SSH**

Always allow SSH BEFORE you enable the firewall, or you lock yourself out. Then ufw allow 443/tcp and ufw enable.

```bash
ufw allow 22/tcp
```

**3. Low-level firewall rules**

Lists the real rules ufw writes. -n = numbers only (faster).

```bash
iptables -L -n
```

**4. Capture packets on port 80**

Shows 10 packets live. Used when you must prove whether traffic arrives at all.

```bash
tcpdump -i any port 80 -c 10
```

**5. SSH tunnel to a remote port**

Makes the server's port 80 appear as localhost:8080 on your machine.

```bash
ssh -L 8080:localhost:80 user@server
```

## Interview Q&A

**1. A website does not open. What do you check, in order?**

- 1 Network: ping the IP (8.8.8.8) - do I have internet?
- 2 DNS: nslookup the name - does it give an IP?
- 3 Port: nc -zv host 443 - is the port open?
- 4 App: curl -I https://host - what HTTP code comes back?

**2. What happens when you type a URL in the browser?**

- Short answer: DNS finds the IP, TCP connects (3-way handshake), TLS sets up encryption for HTTPS, the browser sends an HTTP request, the server answers, the browser renders it.

**3. Explain the TCP 3-way handshake**

- Steps: SYN (client asks), SYN-ACK (server agrees), ACK (client confirms). Then data flows.

**4. What does /24 mean (192.168.1.0/24)?**

- Meaning: The first 24 bits are the network, the last 8 are hosts: 256 addresses (254 usable). /16 = 65,536, /32 = one address.

**5. TCP vs UDP?**

- TCP: Reliable, ordered, connection first. Web, SSH, databases.
- UDP: Fast, no guarantee. DNS lookups, video, games.

**6. Connection refused vs timed out?**

- Refused: The machine answered: nothing listens on that port (service down).
- Timed out: No answer at all: firewall or wrong route.

**7. Which process uses port 80?**

- Answer: ss -tlnp | grep :80 (or lsof -i :80).

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
pkill nc
rm -rf ~/lab

# 4 · NGINX

Install and start NGINX, serve a page, then reverse proxy, load balance and debug a 502. Config files are ready-made, so you only copy, test and reload. Uses its own ports (8000-8083).

## Level 1 · Basic

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Setup: refresh the package list**

Refreshes the list of packages. Needs internet the first time. Skip it if NGINX is already installed.

In [ ]:
%%b
apt-get update -qq

**2. Setup: install NGINX**

On a real server: sudo apt install nginx. It installs the program, the config in /etc/nginx and the logs in /var/log/nginx.

In [ ]:
%%b
apt-get install -y -qq nginx

**Colab only:** the install may already have started a default NGINX on port 80. Stop it so ours starts cleanly in a few steps.

In [ ]:
%%b
service nginx stop

**3. Which version?**

Prints the version. -V (capital) also prints the build options and modules.

In [ ]:
%%b
nginx -v

**4. Copy a small site config**

site.conf says: listen on port 8000 and serve the files in /var/www/lab. Read it with cat. Every file in conf.d/ is included automatically.

In [ ]:
%%b
cp /scripts/lab-files/site.conf /etc/nginx/conf.d/site.conf

**5. Make the web folder**

The 'web root': the folder NGINX serves files from.

In [ ]:
%%b
mkdir -p /var/www/lab

**6. Create a home page**

index.html is what NGINX shows for the path /.

In [ ]:
%%b
echo 'My first site' > /var/www/lab/index.html

**7. Test the config**

ALWAYS run this before starting or reloading. It checks the syntax and does not touch the running server.

In [ ]:
%%b
nginx -t

**8. Start NGINX**

Starts the server in the background. On a real server you use systemctl start nginx (a container has no systemd). If you see 'Address already in use', NGINX is already running: use nginx -s reload.

In [ ]:
%%b
nginx

**9. Open the page**

curl is the quickest way to test a web server from the terminal. Your page came from /var/www/lab.

In [ ]:
%%b
curl localhost:8000

**10. Look at the response header**

200 OK. Other codes: 301 redirect, 403 no permission, 404 file not found, 502 backend problem.

In [ ]:
%%b
curl -I localhost:8000

**11. Where is the config?**

nginx.conf is the main file. conf.d/ holds extra files that are included automatically. sites-enabled/ holds site configs on Ubuntu.

In [ ]:
%%b
ls /etc/nginx

**12. Where are the logs?**

access.log = every request. error.log = problems. When something breaks, read error.log first.

In [ ]:
%%b
ls /var/log/nginx

**13. Read the last request**

Each line: client IP, time, request, status code, bytes, browser. Your curl commands are in there.

In [ ]:
%%b
tail -2 /var/log/nginx/access.log

## Level 2 · Next level

*Run Level 1 first.*

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Change the page**

Changing a FILE needs no reload. Only config changes do.

In [ ]:
%%b
echo 'Version 2' > /var/www/lab/index.html

**2. See the change**

NGINX reads the file on every request.

In [ ]:
%%b
curl localhost:8000

**3. Copy a reverse proxy config**

proxy.conf has 2 servers: a 'backend' on port 9000 and a proxy on port 8080 that forwards to it. Read it with cat.

In [ ]:
%%b
cp /scripts/lab-files/proxy.conf /etc/nginx/conf.d/lab.conf

**4. Test the config**

Syntax ok = safe to reload.

In [ ]:
%%b
nginx -t

**5. Reload without downtime**

-s reload applies the new config and keeps existing connections alive. On a real server: systemctl reload nginx. Restart drops connections, reload does not.

In [ ]:
%%b
nginx -s reload

**6. Call the backend directly**

This is the application server (the 'upstream'). Normally users never reach it directly.

In [ ]:
%%b
curl localhost:9000

**7. Call it through the proxy**

Same answer, but NGINX forwarded the request with proxy_pass. A reverse proxy sits in front of apps and gives you TLS, caching, load balancing and one public entry point.

In [ ]:
%%b
curl localhost:8080

**8. See the requests in the log**

Every request through NGINX is logged. First place to look when users report errors.

In [ ]:
%%b
tail -3 /var/log/nginx/access.log

## Level 3 · Medium

*Run Level 1 first.*

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab
cd ~/lab

**1. Copy a load balancer config**

lb.conf defines an upstream pool with 2 backends (ports 9001 and 9002) and a front server on port 8081.

In [ ]:
%%b
cp /scripts/lab-files/lb.conf /etc/nginx/conf.d/lb.conf

**2. Test the config**

Always test before reload.

In [ ]:
%%b
nginx -t

**3. Reload**

Applies lb.conf.

In [ ]:
%%b
nginx -s reload

**4. Request 1**

The answer says which backend served you.

In [ ]:
%%b
curl localhost:8081

**5. Request 2**

It should be the OTHER backend: the default method is round robin (take turns). Others: least_conn (fewest active connections), ip_hash (same client, same backend).

In [ ]:
%%b
curl localhost:8081

**6. Request 3**

Back to the first backend. If one backend dies, NGINX stops sending traffic to it.

In [ ]:
%%b
curl localhost:8081

**7. Copy a broken proxy config**

bad.conf forwards to port 9999, where nothing is running. We create a 502 on purpose.

In [ ]:
%%b
cp /scripts/lab-files/bad.conf /etc/nginx/conf.d/bad.conf

**8. Reload**

Applies bad.conf.

In [ ]:
%%b
nginx -s reload

**9. See the error code**

502 = NGINX is fine but could not get a valid answer from the backend. 504 = the backend answered too slowly (timeout).

In [ ]:
%%b
curl -I localhost:8082

**10. Find the reason in the error log**

connect() failed (111: Connection refused) = nothing listens on that port. Fix the backend or the port number. Interview: '502, what do you check?' error.log first, then is the backend running.

In [ ]:
%%b
tail -2 /var/log/nginx/error.log

**11. Copy a routing config**

routes.conf shows location blocks: a prefix /api/, an exact /health and a default /.

In [ ]:
%%b
cp /scripts/lab-files/routes.conf /etc/nginx/conf.d/routes.conf

**12. Reload**

Applies routes.conf.

In [ ]:
%%b
nginx -s reload

**13. Test the health route**

location = /health is an EXACT match. Health checks use a route like this.

In [ ]:
%%b
curl localhost:8083/health

**14. Test the api route**

location /api/ is a prefix match: anything that starts with /api/ lands here.

In [ ]:
%%b
curl localhost:8083/api/users

**15. Stop NGINX**

Fast shutdown. -s quit finishes current requests first.

In [ ]:
%%b
nginx -s stop

## On a real server (TLS, limits) (reference)

Needs a certificate and systemd. Copy-only.

**1. Start, reload, restart as a service**

Use reload for config changes, restart only when needed.

```bash
systemctl reload nginx
```

**2. TLS termination (inside a server block)**

NGINX handles HTTPS and talks plain HTTP to the backend. Put these lines in the server block.

```bash
listen 443 ssl; ssl_certificate /etc/ssl/site.crt; ssl_certificate_key /etc/ssl/site.key;
```

**3. Redirect http to https**

Put this in a server block that listens on port 80.

```bash
return 301 https://$host$request_uri;
```

**4. Rate limiting (http block)**

Defines a zone: max 10 requests per second per client IP. Then limit_req zone=one; inside a location. Extra requests get 503.

```bash
limit_req_zone $binary_remote_addr zone=one:10m rate=10r/s;
```

**5. Pass the real client IP to the app**

Without it the backend only sees NGINX's address.

```bash
proxy_set_header X-Real-IP $remote_addr;
```

## Interview Q&A

**1. What is a reverse proxy?**

- Answer: A server in front of your apps. Clients talk to it, it forwards to the real servers. Gives TLS, caching, load balancing and one entry point.
- Forward proxy: Sits in front of CLIENTS (like a company proxy for outgoing traffic). Opposite direction.

**2. 502 vs 504?**

- 502 Bad Gateway: The backend is down, refused the connection or sent garbage. Check error.log and whether the backend runs.
- 504 Gateway Timeout: The backend is up but too slow. Check backend load or raise proxy_read_timeout.

**3. Reload vs restart?**

- Reload: Re-reads the config, workers finish old requests, no downtime.
- Restart: Stops everything and starts again: brief downtime.

**4. Name the load balancing methods**

- Default: Round robin (take turns).
- Others: least_conn, ip_hash (sticky by client), weights.

**5. How do you check a config change safely?**

- Answer: nginx -t, then nginx -s reload. Never reload without testing.

**6. Where do you look when a site is broken?**

- Logs: /var/log/nginx/error.log and access.log.
- Also: Is NGINX running? Is the port listening (ss -tln)? Does the backend answer (curl localhost:PORT)?

**7. What does location / vs location = /health mean?**

- Prefix: location /api/ matches everything that starts with /api/.
- Exact: location = /health matches only that one path.

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
nginx -s stop
rm -f /etc/nginx/conf.d/site.conf /etc/nginx/conf.d/lab.conf /etc/nginx/conf.d/lb.conf /etc/nginx/conf.d/bad.conf /etc/nginx/conf.d/routes.conf
rm -rf /var/www/lab ~/lab

# 5 · Docker (Week 2)

Images and containers first, then inspecting and debugging running containers, then your own image, volumes and networks. Runs in the toolbox terminal that talks to Docker Desktop.

> Docker cannot run inside Colab (there is no Docker engine). Practise this topic in the lab page on Docker Desktop (http://localhost:7000/ssh-lab.html, topic 5). The commands and notes are here for reference.

## Level 1 · Basic

**1. Is Docker there?**

Prints the client version. If later commands hang, the Docker engine (Docker Desktop) is not running.

```bash
docker --version
```

**2. Download an image**

An IMAGE is a read-only package: an app and its files. pull downloads it from Docker Hub. alpine is a tiny 5 MB Linux. No tag means :latest.

```bash
docker pull alpine
```

**3. List images**

Shows repository, tag, id and size of every image on this machine.

```bash
docker images
```

**4. Run a container**

A CONTAINER is a running copy of an image. run = create + start. The container ran echo, then ended because its command finished.

```bash
docker run --name lab-hi alpine echo hello
```

**5. List running containers**

Only RUNNING containers. lab-hi is not listed because it already exited.

```bash
docker ps
```

**6. List all containers**

-a includes stopped containers. Status 'Exited (0)' = finished normally. Interview: 'container exits immediately, why?' Its main command finished or crashed: look at the exit code and docker logs.

```bash
docker ps -a
```

**7. Remove it**

Deletes a stopped container. The image stays.

```bash
docker rm lab-hi
```

**8. Run a web server in the background**

-d = detached (background). --name gives it a name. -p 8097:80 = host port 8097 to container port 80. Prints the container id.

```bash
docker run -d --name lab-web -p 8097:80 nginx:1.27-alpine
```

**9. Is it running?**

Now you see it with its ports and status 'Up'.

```bash
docker ps
```

**10. Read its logs**

logs shows what the container printed on stdout and stderr. -f follows live, --tail 20 shows the last 20 lines. First step when a container misbehaves.

```bash
docker logs lab-web
```

**11. Stop it**

Sends SIGTERM (polite), waits 10 seconds, then SIGKILL. docker start lab-web brings the same container back.

```bash
docker stop lab-web
```

**12. Delete it**

Removes the stopped container. docker rm -f does stop and remove in one go.

```bash
docker rm lab-web
```

## Level 2 · Next level

**1. Start a web server again**

Same command as before. This level inspects a running container.

```bash
docker run -d --name lab-web -p 8097:80 nginx:1.27-alpine
```

**2. Run one command inside it**

exec runs a command inside a RUNNING container. For a shell use docker exec -it lab-web sh (-i keep input open, -t terminal). Type exit to leave.

```bash
docker exec lab-web nginx -v
```

**3. Look around inside**

The container has its own file system, taken from the image.

```bash
docker exec lab-web ls /etc/nginx
```

**4. Last log lines**

--tail N limits the output. Use it on chatty containers.

```bash
docker logs --tail 3 lab-web
```

**5. Ask for one fact**

inspect prints all the details as JSON: ip, mounts, env, restart policy. --format picks one field.

```bash
docker inspect lab-web --format '{{.State.Status}}'
```

**6. Check the port mapping**

Shows which host port maps to which container port.

```bash
docker port lab-web
```

**7. CPU and memory**

Live usage per container. --no-stream prints once.

```bash
docker stats --no-stream
```

**8. Processes inside**

Like ps for one container.

```bash
docker top lab-web
```

**9. Restart it**

stop + start. Used after changing something the container reads only at startup.

```bash
docker restart lab-web
```

**10. Stop and remove in one go**

-f forces removal of a running container.

```bash
docker rm -f lab-web
```

## Level 3 · Medium

**1. Read the sample Dockerfile**

FROM = base image. RUN = a command executed while BUILDING (saved as a layer). CMD = the default command when a container STARTS. Mixing up RUN and CMD is the most common beginner mistake.

```bash
cat Dockerfile
```

**2. Build your image**

build reads the Dockerfile in the folder (the dot = build context). -t names the image.

```bash
docker build -t lab-hello .
```

**3. Run your image**

--rm deletes the container when it exits, so nothing piles up.

```bash
docker run --rm lab-hello
```

**4. Build again**

Nothing changed, so every layer comes from the cache. If you change one line, that layer and all layers AFTER it rebuild: put rarely changing steps first.

```bash
docker build -t lab-hello .
```

**5. See the layers**

Every Dockerfile instruction is one layer. Smaller and fewer layers = smaller image.

```bash
docker history lab-hello
```

**6. Create a volume**

A volume keeps data OUTSIDE the container, so it survives container deletion. Files written inside a container are lost when it is removed.

```bash
docker volume create lab-vol
```

**7. Write a file into the volume**

-v lab-vol:/data mounts the volume at /data. The container is deleted afterwards.

```bash
docker run --rm -v lab-vol:/data alpine touch /data/f.txt
```

**8. Read it from a NEW container**

The file is still there: that is persistence. Databases must always use a volume.

```bash
docker run --rm -v lab-vol:/data alpine ls /data
```

**9. Create a network**

A user-defined network gives containers DNS: they can reach each other by NAME.

```bash
docker network create lab-net
```

**10. Start a web container on it**

No -p needed: other containers on the same network reach it directly.

```bash
docker run -d --name lab-web --network lab-net nginx:1.27-alpine
```

**11. Reach it by name from another container**

The name lab-web resolved by itself. This is how Compose services find each other (api talks to db by name).

```bash
docker run --rm --network lab-net alpine ping -c 1 lab-web
```

**12. Clean up the container**

Remove the running container.

```bash
docker rm -f lab-web
```

**13. Clean up the image**

rmi removes an image. docker system prune removes all unused things (ask first!).

```bash
docker rmi lab-hello
```

**14. Clean up the volume and network**

Volumes are never deleted automatically.

```bash
docker volume rm lab-vol
```

**15. Remove the network**

Everything created by this topic is now gone.

```bash
docker network rm lab-net
```

## Interactive and more (reference)

These need a live terminal session. Copy and run them in the terminal on the right.

**1. Open a shell inside a container**

-it = interactive terminal. Type exit to leave. The container keeps running.

```bash
docker exec -it lab-web sh
```

**2. Run a throwaway container with a shell**

--rm deletes it on exit. Perfect for trying commands safely.

```bash
docker run -it --rm alpine sh
```

**3. Free disk space**

Removes stopped containers, unused networks and dangling images. It asks first.

```bash
docker system prune
```

**4. See disk used by Docker**

Images, containers, volumes and build cache, and how much can be reclaimed.

```bash
docker system df
```

## Interview Q&A

**1. Image vs container?**

- Image: Read-only package: app + files. Built once.
- Container: A running (or stopped) instance of an image. You can start many from one image.

**2. A container exits immediately. Why?**

- Reason: Its main process (PID 1) ended or crashed. A web server stays up because it never ends.
- Check: docker ps -a (exit code), then docker logs NAME.

**3. CMD vs ENTRYPOINT?**

- ENTRYPOINT: The fixed program.
- CMD: The default arguments. docker run IMAGE args replaces CMD, not ENTRYPOINT.

**4. COPY vs ADD?**

- Rule: Use COPY. ADD can also download URLs and unpack archives, which surprises people.

**5. Volume vs bind mount?**

- Volume: Managed by Docker, best for databases and persistent data.
- Bind mount: A host folder mapped into the container, good for source code in development.

**6. EXPOSE vs -p?**

- EXPOSE: Documentation only: the image uses this port.
- -p 8080:80: Actually publishes the port on the host.

**7. How do you make an image smaller?**

- Ideas: Small base (alpine), multi-stage builds, fewer layers, .dockerignore, no build tools in the final image.

**8. How do two containers talk to each other?**

- Answer: Put them on the same user-defined network and use the container name as the host name (Docker's built-in DNS).

# 6 · Git and CI/CD (Week 3)

Git first (commit, branch, merge), then read a real GitHub Actions workflow file and ask it questions. The pipeline itself runs on GitHub, so the last tab is copy-only.

**Colab setup:** installs `yq`, the tool used to read the workflow file later. (The lab page already has it.)

In [ ]:
%%b
wget -q -O /usr/local/bin/yq https://github.com/mikefarah/yq/releases/download/v4.44.3/yq_linux_amd64
chmod +x /usr/local/bin/yq
yq --version

## Level 1 · Basic

**1. Make a practice folder**

All git practice stays inside ~/lab.

In [ ]:
%%b
mkdir -p ~/lab

**2. Go into it**

cd = change directory.

In [ ]:
%%b
cd ~/lab

**3. Create a repository**

git init turns a folder into a repository: it adds a hidden .git folder that stores the whole history. -b main names the first branch main.

In [ ]:
%%b
git init -b main app

**4. Go into the repository**

Everything from here on happens inside app.

In [ ]:
%%b
cd app

**5. Tell git who you are**

Every commit records an author. Without --global this applies to this repository only.

In [ ]:
%%b
git config user.name 'Lab User'

**6. And your email**

Use your real email on a real project: GitHub matches commits to your account with it.

In [ ]:
%%b
git config user.email 'lab@example.com'

**7. Turn off the pager**

By default git opens long output in a pager (press q to quit). cat just prints it, which is easier here.

In [ ]:
%%b
git config core.pager cat

**8. Create a file**

A new file. Git does not track it yet.

In [ ]:
%%b
echo hello > app.txt

**9. Check the state**

git status is the command you will type most. 'Untracked' = git sees the file but is not watching it.

In [ ]:
%%b
git status

**10. Stage the file**

git add puts the change in the staging area: the list of changes that go into the next commit. git add . stages everything.

In [ ]:
%%b
git add app.txt

**11. Save a commit**

A commit is a snapshot with a message. -m gives the message. Write what changed and why.

In [ ]:
%%b
git commit -m 'first commit'

**12. See the history**

One line per commit: short id and message. HEAD is where you are now.

In [ ]:
%%b
git log --oneline

**13. Check again**

'nothing to commit, working tree clean' = everything is saved.

In [ ]:
%%b
git status

## Level 2 · Next level

*Run Level 1 first.*

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab/app
cd ~/lab/app

**1. Start a new branch**

A branch is a separate line of work. -c creates it and switches. Real teams make one branch per change and open a pull request.

In [ ]:
%%b
git switch -c feature

**2. Change the file**

>> adds a line at the end.

In [ ]:
%%b
echo change >> app.txt

**3. See what changed**

git diff shows changes that are NOT staged yet. + lines were added, - lines removed. git diff --staged shows what is staged.

In [ ]:
%%b
git diff

**4. Stage it**

Staging again.

In [ ]:
%%b
git add app.txt

**5. Commit it**

The commit goes on the feature branch only.

In [ ]:
%%b
git commit -m 'add change'

**6. Look at the history**

Two commits now. The newest is on top.

In [ ]:
%%b
git log --oneline

**7. Go back to main**

main does not have the change yet.

In [ ]:
%%b
git switch main

**8. Merge the branch**

merge brings the feature commits into main. Fast-forward = main had no new commits, so git just moves the pointer. Otherwise git makes a merge commit and conflicts may appear.

In [ ]:
%%b
git merge feature

**9. List branches**

The star marks the branch you are on.

In [ ]:
%%b
git branch

**10. Delete the finished branch**

-d only deletes if it is merged. -D forces it.

In [ ]:
%%b
git branch -d feature

## Level 3 · Medium

*Run Level 1 first.*

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/lab/app
cd ~/lab/app

**1. Ignore log files**

.gitignore lists files git must never track: logs, build output, secrets like .env.

In [ ]:
%%b
echo '*.log' > .gitignore

**2. Create a log file**

This file matches the pattern, so git will ignore it.

In [ ]:
%%b
touch debug.log

**3. Check what git sees**

--short = one line per file. ?? = untracked. debug.log is not listed because it is ignored.

In [ ]:
%%b
git status --short

**4. Make the workflow folder**

GitHub runs every YAML file in .github/workflows. The folder name is fixed.

In [ ]:
%%b
mkdir -p .github/workflows

**5. Copy a sample workflow**

A small CI pipeline: on every push to main (and every pull request) it checks out the code, runs tests and builds an image.

In [ ]:
%%b
cp /work/lab-ci/ci.yml .github/workflows/ci.yml

**6. Read it**

name = the workflow name. on = triggers. jobs = parallel groups of work. runs-on = the machine type. steps = commands in order. uses = a ready-made action, run = a shell command.

In [ ]:
%%b
cat .github/workflows/ci.yml

**7. Ask it: when does it run?**

yq reads YAML like jq reads JSON. The triggers are push to main and any pull request.

In [ ]:
%%b
yq '.on' .github/workflows/ci.yml

**8. Ask it: what are the steps?**

The steps run top to bottom. If one fails, the job stops and the pull request shows a red cross.

In [ ]:
%%b
yq '.jobs.build.steps[].name' .github/workflows/ci.yml

**9. Stage everything**

Stages the new files. debug.log is skipped.

In [ ]:
%%b
git add .

**10. Commit the workflow**

Pushing this file to GitHub is all it takes to start the pipeline.

In [ ]:
%%b
git commit -m 'add CI workflow'

**11. See the history**

Your pipeline is now versioned like any other code.

In [ ]:
%%b
git log --oneline

**12. Tag a release**

A tag is a permanent name for a commit, used for releases. A workflow can trigger on tags.

In [ ]:
%%b
git tag v1.0

**13. List tags**

Tags follow semantic versioning: v1.0.0 (major.minor.patch).

In [ ]:
%%b
git tag

## On GitHub (copy only) (reference)

These live in the workflow file or run on GitHub. Copy them into ci.yml when you want to try them.

**1. Run on a schedule or by hand**

workflow_dispatch adds a 'Run workflow' button. schedule uses cron syntax (UTC).

```bash
on: { workflow_dispatch: {}, schedule: [{ cron: '0 2 * * *' }] }
```

**2. Run the job on several versions (matrix)**

One job definition runs once per value, in parallel. Use ${{ matrix.node }} inside the steps.

```bash
strategy: { matrix: { node: [18, 20] } }
```

**3. Use a secret**

Secrets are set in the repository settings. They are masked in the logs. Never print them or commit them.

```bash
env: { TOKEN: '${{ secrets.MY_TOKEN }}' }
```

**4. Cache dependencies**

Saves a folder (node_modules, ~/.m2) between runs so builds are faster. The key decides when the cache is reused.

```bash
uses: actions/cache@v4
```

**5. Keep a build result**

Saves files (reports, binaries) so you can download them or use them in a later job.

```bash
uses: actions/upload-artifact@v4
```

**6. Log in to AWS without a stored key (OIDC)**

OIDC lets GitHub prove its identity to the cloud and get a short-lived token. No long-lived access keys in your secrets.

```bash
permissions: { id-token: write, contents: read }
```

**7. Run one job after another**

By default jobs run in parallel. needs makes a job wait for another one.

```bash
needs: build
```

## Interview Q&A

**1. Workflow, job, step: what is the difference?**

- Workflow: One YAML file. Starts on an event (push, pull request, schedule).
- Job: A group of steps that runs on one machine. Jobs run in parallel unless one needs another.
- Step: One command (run) or one ready-made action (uses).

**2. What is CI and what is CD?**

- CI: Continuous Integration: every push is built and tested automatically, so broken code is found early.
- CD: Continuous Delivery/Deployment: a tested build is released automatically (delivery stops at a manual approval, deployment goes all the way).

**3. How do you keep secrets safe in a pipeline?**

- Answer: Store them as encrypted secrets, never in the repository. Prefer OIDC (short-lived tokens) over long-lived keys. Give each job the least permission it needs.

**4. What is a matrix build?**

- Answer: One job that runs several times with different values (OS, language version), in parallel.

**5. How do you make a pipeline faster?**

- Ideas: Cache dependencies, run independent jobs in parallel, build only what changed, use a smaller base image.

**6. Merge vs rebase?**

- Merge: Keeps all history and adds a merge commit.
- Rebase: Replays your commits on top of the target for a straight, clean history. Never rebase commits that others already use.

**7. How do you undo things in git?**

- Not committed: git restore FILE discards changes to a file.
- Committed: git revert COMMIT makes a new commit that undoes it (safe for shared branches). git reset moves the branch back (rewrites history).

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
rm -rf ~/lab

# 7 · Terraform (Week 3)

Write, plan and apply infrastructure as code. The lab uses Terraform's built-in terraform_data resource, so it needs no cloud account and no internet: same workflow, nothing to pay for.

**Colab setup:** installs Terraform. (The lab page already has it.)

In [ ]:
%%b
apt-get install -y -qq unzip
cd /tmp
wget -q https://releases.hashicorp.com/terraform/1.10.3/terraform_1.10.3_linux_amd64.zip
unzip -qo terraform_1.10.3_linux_amd64.zip
mv terraform /usr/local/bin/
terraform version

## Level 1 · Basic

**1. Make a project folder**

A Terraform project is just a folder of .tf files.

In [ ]:
%%b
mkdir -p ~/tf

**2. Go into it**

Terraform always works on the files in the current folder.

In [ ]:
%%b
cd ~/tf

**3. Which version?**

Version of the tool. Teams pin it, because state files from a newer version cannot be read by an older one.

In [ ]:
%%b
terraform version

**4. Copy a small configuration**

main.tf is the usual name for the main file.

In [ ]:
%%b
cp /work/lab-tf/basic.tf main.tf

**5. Read it**

variable = an input with a default. resource = a thing Terraform manages (type, then your name). output = a value printed at the end. Cloud resources look the same, for example aws_s3_bucket.

In [ ]:
%%b
cat main.tf

**6. Initialise**

init downloads providers and sets up the backend. Run it first in every new folder, and again after you add a provider or module.

In [ ]:
%%b
terraform init

**7. Check the syntax**

validate checks the files for mistakes without touching anything.

In [ ]:
%%b
terraform validate

**8. See the plan**

plan compares your code with reality and shows what WOULD change. + create, ~ change, - destroy. Always read it before applying.

In [ ]:
%%b
terraform plan

**9. Apply it**

apply makes the change. -auto-approve skips the yes/no question: fine in a lab, in real life review the plan first.

In [ ]:
%%b
terraform apply -auto-approve

**10. Show the outputs**

Outputs are the values you want to read afterwards: an IP address, a URL.

In [ ]:
%%b
terraform output

**11. List what Terraform manages**

The state file is Terraform's memory of what it created. Without it Terraform would not know what exists.

In [ ]:
%%b
terraform state list

## Level 2 · Next level

*Run Level 1 first.*

*Go to the practice folder first.*

In [ ]:
%%b
mkdir -p ~/tf
cd ~/tf

**1. Plan again**

Nothing changed, so nothing to do. This is idempotence: applying twice gives the same result.

In [ ]:
%%b
terraform plan

**2. Change an input variable**

-var overrides a variable for one run. Real projects use a terraform.tfvars file instead.

In [ ]:
%%b
terraform apply -auto-approve -var name=Asha

**3. Read the new output**

Naming the output prints just its value, handy in scripts.

In [ ]:
%%b
terraform output greeting

**4. Format the code**

fmt fixes indentation and spacing. It prints the names of files it changed. No output = already tidy. CI pipelines run terraform fmt -check.

In [ ]:
%%b
terraform fmt

**5. Check the syntax again**

Run validate and fmt before every commit.

In [ ]:
%%b
terraform validate

**6. Preview another change**

A plan is only a preview. Nothing changes until apply.

In [ ]:
%%b
terraform plan -var name=Ravi

**7. Look at the state file**

terraform.tfstate holds the real values. It can contain secrets, so never commit it to git. Teams keep it in a remote backend (S3 + DynamoDB lock, or Azure Storage) so two people cannot apply at once.

In [ ]:
%%b
ls

**8. Inspect one resource**

Shows everything Terraform knows about one resource.

In [ ]:
%%b
terraform state show terraform_data.hello

**9. Destroy everything**

destroy removes every resource in the state. In real life, be very careful.

In [ ]:
%%b
terraform destroy -auto-approve

**10. Check that it is empty**

No output = nothing is managed any more.

In [ ]:
%%b
terraform state list

## Level 3 · Medium

**1. Make a second project**

A fresh folder for a project that uses a module.

In [ ]:
%%b
mkdir -p ~/tf2

**2. Go into it**

cd = change directory.

In [ ]:
%%b
cd ~/tf2

**3. Copy the project**

Copies main.tf and a modules folder. The dot at the end means 'into this folder'.

In [ ]:
%%b
cp -r /work/lab-tf/mod/. .

**4. See the layout**

A module is just a folder with .tf files that you reuse, like a function.

In [ ]:
%%b
ls

**5. Read the root file**

The module block calls the module: source = where it is, names = the input it needs.

In [ ]:
%%b
cat main.tf

**6. Read the module**

for_each creates one resource per item in the list. count = N is the simpler cousin. Prefer for_each when items have names, because removing one does not renumber the others.

In [ ]:
%%b
cat modules/greeter/main.tf

**7. Initialise**

init also loads the modules this time.

In [ ]:
%%b
terraform init

**8. Plan**

Three names in the list = three resources.

In [ ]:
%%b
terraform plan

**9. Apply**

All three are created.

In [ ]:
%%b
terraform apply -auto-approve

**10. Show the output**

The module returned a list; the root file printed it.

In [ ]:
%%b
terraform output

**11. See the addresses**

module.NAME.TYPE.NAME[key] is the full address of a resource inside a module.

In [ ]:
%%b
terraform state list

**12. Clean up**

Removes the three resources.

In [ ]:
%%b
terraform destroy -auto-approve

## With a real cloud (copy only) (reference)

These need a cloud account. Copy them when you practise on a KodeKloud AWS or Azure playground.

**1. Store state remotely with locking (AWS)**

State goes into an S3 bucket and DynamoDB blocks two people from applying at the same time. Azure uses a storage account with blob lease locking.

```bash
backend "s3" { bucket = "my-state" key = "app/terraform.tfstate" region = "us-east-1" dynamodb_table = "tf-lock" }
```

**2. Move local state to the backend**

After adding a backend block, this copies your existing state into it.

```bash
terraform init -migrate-state
```

**3. A real resource**

Same shape as the lab resource. The provider (aws) talks to the cloud API.

```bash
resource "aws_s3_bucket" "logs" { bucket = "my-unique-logs-bucket" }
```

**4. Bring an existing resource under Terraform**

import adds a resource that was created by hand to the state. You still have to write the matching code.

```bash
terraform import aws_s3_bucket.logs my-unique-logs-bucket
```

**5. Run it in a pipeline**

Save the plan in CI, show it on the pull request, and apply that exact plan after approval: terraform apply tfplan.

```bash
terraform plan -out=tfplan
```

## Interview Q&A

**1. What is the Terraform state and why is it important?**

- Answer: The file where Terraform records what it created. It maps your code to real resources. Lose it and Terraform no longer knows what exists.
- Teams: Keep it in a remote backend with locking, encrypted, never in git.

**2. terraform plan vs apply?**

- plan: Preview only. Compares code, state and reality.
- apply: Makes the changes. In CI you apply the saved plan after approval.

**3. What is drift and how do you handle it?**

- Drift: Someone changed the real infrastructure by hand, so it no longer matches the code.
- Fix: terraform plan shows the difference. Apply to put it back, or update the code to match.

**4. count vs for_each?**

- count: N copies, addressed by number (0, 1, 2). Removing item 0 shifts the others.
- for_each: One per key in a map or set, addressed by name. Safer for changes.

**5. What is a module?**

- Answer: A reusable folder of Terraform code with inputs (variables) and outputs. Used to avoid copy and paste, for example one VPC module for every environment.

**6. How do you handle secrets?**

- Answer: Mark variables sensitive = true, pass them from environment variables or a secret manager, and keep the state encrypted. They still end up in state, so protect the state.

**7. Terraform vs Ansible?**

- Terraform: Creates infrastructure (servers, networks). Declarative, keeps state.
- Ansible: Configures what runs ON the servers (packages, files). Usually no state.

### Reset this topic (optional)

Removes what this topic created so you can start over.

In [ ]:
%%b
cd ~
rm -rf ~/tf ~/tf2

# 8 · Cloud AWS / Azure (Week 3)

Cloud basics you are asked about: who am I, storage, servers, networks and permissions. There is no cloud account in the lab, so the commands are copy-only for a KodeKloud playground. The Q&A tab is the part to learn.

> This topic needs a cloud account (AWS or Azure), so the commands are for reference. Run them in a KodeKloud playground.

## Level 1 · Basic (reference)

**1. Who am I? (AWS)**

Shows the account number and the user or role you are using. First command when something says 'access denied'.

```bash
aws sts get-caller-identity
```

**2. Who am I? (Azure)**

Shows the subscription and tenant. az login signs you in.

```bash
az account show
```

**3. List storage buckets (AWS S3)**

S3 = object storage: files in buckets. Cheap, durable, used for backups, logs, websites and Terraform state.

```bash
aws s3 ls
```

**4. Upload a file to S3**

cp copies between local and S3. aws s3 sync syncs whole folders.

```bash
aws s3 cp notes.txt s3://my-bucket/
```

**5. List servers (AWS EC2)**

EC2 = virtual servers. --query filters the JSON output.

```bash
aws ec2 describe-instances --query 'Reservations[].Instances[].InstanceId'
```

**6. List resource groups (Azure)**

A resource group is a folder for related resources. -o table = readable output.

```bash
az group list -o table
```

**7. List VMs (Azure)**

Azure virtual machines.

```bash
az vm list -o table
```

## Level 2 · Next level (reference)

**1. List IAM users**

IAM = who may do what. A USER is a person, a ROLE is an identity that services or people assume temporarily. Prefer roles.

```bash
aws iam list-users
```

**2. List IAM roles**

Roles hold permissions through policies and give short-lived credentials.

```bash
aws iam list-roles
```

**3. List VPCs**

A VPC is your private network in the cloud, with its own IP range (CIDR, for example 10.0.0.0/16).

```bash
aws ec2 describe-vpcs
```

**4. List subnets**

Subnets split the VPC. Public subnet = has a route to an internet gateway. Private subnet = no direct internet, uses a NAT gateway to go out.

```bash
aws ec2 describe-subnets
```

**5. List security groups**

A security group is a firewall around a server. Stateful: if a request is allowed in, the reply is allowed out.

```bash
aws ec2 describe-security-groups
```

**6. List Azure networks**

VNet = Azure's VPC. NSG = network security group, the firewall.

```bash
az network vnet list -o table
```

**7. List Azure role assignments**

Azure RBAC: who has which role on which scope.

```bash
az role assignment list -o table
```

## Level 3 · Medium (reference)

**1. Log in to the container registry (AWS ECR)**

ECR stores your Docker images. The password is piped to docker login. Azure's version is ACR: az acr login --name NAME.

```bash
aws ecr get-login-password
```

**2. Read application logs (CloudWatch)**

CloudWatch Logs collects logs and metrics. Azure Monitor is the equivalent.

```bash
aws logs tail /aws/lambda/my-function --follow
```

**3. List alarms**

Alarms notify you when a metric (CPU, errors) crosses a limit.

```bash
aws cloudwatch describe-alarms
```

**4. Assume a role**

Returns temporary keys. This is the safe way to get higher access for a short time.

```bash
aws sts assume-role --role-arn ARN --role-session-name lab
```

**5. Keyless login from GitHub (idea)**

A GitHub workflow proves who it is with OIDC and assumes a role. No access keys are stored.

```bash
aws-actions/configure-aws-credentials with role-to-assume
```

## Interview Q&A

**1. What is IAM and what is least privilege?**

- IAM: The service that controls who (users, roles) may do what (policies) on which resources.
- Least privilege: Give only the permissions needed for the job, nothing more. Start with none and add.

**2. IAM user vs role?**

- User: A long-lived identity for a person, with keys or a password.
- Role: A temporary identity assumed by a person, service or pipeline. No stored keys. Preferred.

**3. Public vs private subnet?**

- Public: Has a route to an internet gateway. Load balancers and bastion hosts live here.
- Private: No inbound internet. Apps and databases live here and use a NAT gateway for outbound traffic.

**4. What is a NAT gateway?**

- Answer: Lets servers in a private subnet reach the internet (updates, APIs) without the internet being able to reach them.

**5. Security group vs network ACL?**

- Security group: Firewall on the server. Stateful. Allow rules only.
- Network ACL: Firewall on the subnet. Stateless: you must allow replies too. Allow and deny rules.

**6. Region vs availability zone?**

- Region: A geographic area (Mumbai, Frankfurt).
- Availability zone: One or more separate data centres inside a region. Spread servers across zones to survive a failure.

**7. EC2 vs Lambda (VM vs serverless)?**

- EC2: You run a server and pay while it is on. Full control.
- Lambda: You upload a function, the cloud runs it on demand. Pay per run. Good for short, event-driven work.

**8. How does a pipeline log in to the cloud safely?**

- Answer: OIDC: the pipeline gets a short-lived token and assumes a role. No long-lived access keys in secrets.